# Saved Model Artifact Analysis

Upload a `*_models_and_results_*.zip` archive created by a Keras training notebook to inspect its saved models, training histories, and evaluation metrics. The archive does not include feature data, so re-evaluation and new predictions require loading the matching dataset separately.

In [ ]:
from pathlib import Path
import pickle
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from keras.models import load_model
from google.colab import files


## Load an archive

Set `ARTIFACT_ARCHIVE` to a ZIP already in `/content`, or leave it as `None` to upload one.

In [ ]:
ARTIFACT_ARCHIVE = None  # Example: "/content/cnn_models_and_results_mfcc10.zip"
EXTRACT_ROOT = Path("/content/artifact_analysis")

archive_path = Path(ARTIFACT_ARCHIVE) if ARTIFACT_ARCHIVE else None
if archive_path is None:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one artifact ZIP archive.")
    archive_path = Path("/content") / next(iter(uploaded))
    archive_path.write_bytes(uploaded[archive_path.name])

if not archive_path.is_file() or archive_path.suffix.lower() != ".zip":
    raise FileNotFoundError(f"Artifact ZIP not found: {archive_path}")

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
extract_root_resolved = EXTRACT_ROOT.resolve()
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        try:
            (EXTRACT_ROOT / member.filename).resolve().relative_to(extract_root_resolved)
        except ValueError as exc:
            raise ValueError(f"Unsafe ZIP entry: {member.filename}") from exc
    archive.extractall(EXTRACT_ROOT)

history_paths = sorted(EXTRACT_ROOT.rglob("training_histories_*.pkl"))
if len(history_paths) != 1:
    raise ValueError(f"Expected one history file after extraction, found {len(history_paths)}.")
artifact_dir = history_paths[0].parent
print(f"Analysing artifacts in: {artifact_dir}")


## Evaluation metrics

The training notebook stores separate UUV-focused result CSVs for multilabel and binary models.

In [ ]:
evaluation_frames = []
for result_path in sorted(artifact_dir.glob("*evaluation_results_*.csv")):
    result_frame = pd.read_csv(result_path)
    result_frame.insert(0, "Result file", result_path.stem)
    evaluation_frames.append(result_frame)

if not evaluation_frames:
    raise FileNotFoundError(f"No evaluation CSV files found in {artifact_dir}")

evaluation_results = pd.concat(evaluation_frames, ignore_index=True)
display(evaluation_results)

if "f1-score" in evaluation_results:
    axis = evaluation_results.plot.bar(
        x="Model", y="f1-score", figsize=(12, 5), legend=False, color="steelblue"
    )
    axis.set_ylabel("UUV F1 score")
    axis.set_title("Saved-model UUV F1 scores")
    plt.tight_layout()
    plt.show()


## Training histories

Compare validation loss and ROC AUC for all normal, M-filtered, and W-filtered runs.

In [ ]:
with history_paths[0].open("rb") as history_file:
    histories = pickle.load(history_file)

history_summary_rows = []
for name, history in histories.items():
    history_summary_rows.append({
        "History": name,
        "Epochs": len(history.get("loss", [])),
        "Best validation loss": min(history["val_loss"]) if history.get("val_loss") else None,
        "Best validation ROC AUC": max(history["val_roc_auc"]) if history.get("val_roc_auc") else None,
    })

history_summary = pd.DataFrame(history_summary_rows).sort_values("Best validation ROC AUC", ascending=False)
display(history_summary)

figure, axes = plt.subplots(1, 2, figsize=(16, 5))
for name, history in histories.items():
    if history.get("val_loss"):
        axes[0].plot(history["val_loss"], label=name)
    if history.get("val_roc_auc"):
        axes[1].plot(history["val_roc_auc"], label=name)

axes[0].set_title("Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[1].set_title("Validation ROC AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC AUC")
for axis in axes:
    axis.legend(fontsize=8)

plt.tight_layout()
plt.show()


## Saved Keras models

Load every `.keras` file without recompiling it, then review parameter counts and inspect any selected architecture.

In [ ]:
model_paths = sorted(artifact_dir.glob("*.keras"))
if not model_paths:
    raise FileNotFoundError(f"No .keras model files found in {artifact_dir}")

loaded_models = {model_path.stem: load_model(model_path, compile=False) for model_path in model_paths}
model_summary = pd.DataFrame([
    {
        "Model": name,
        "Parameters": model.count_params(),
        "Input shape": str(model.input_shape),
        "Output shape": str(model.output_shape),
    }
    for name, model in loaded_models.items()
])
display(model_summary.sort_values("Model"))

SELECTED_MODEL = next(iter(loaded_models))  # Replace with a name from the table above.
print(f"Architecture: {SELECTED_MODEL}")
loaded_models[SELECTED_MODEL].summary()
